In [2]:
import asyncio, time
import random
import json
import pandas as pd

In [ ]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
load_dotenv()
client = AsyncOpenAI(base_url="https://openrouter.ai/api/v1",
                api_key=os.environ["OPENROUTER_API_KEY"])

MODEL = "qwen/qwen3.8-flash"
PROMPT = "Name one random animal. Reply with just the word."
SYSTEM_PROMPT = "You are a pirate"

async def real_call(prompt, system_prompt):
    resp = await client.chat.completions.create(
    model= MODEL,
    messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt},
    ],
    temperature=1,
    )
    return resp.choices[0].message.content

print(await real_call(PROMPT, "You are a marine biologist."))

Octopus


In [8]:
sem = asyncio.Semaphore(5)

async def limited_call(i, system_prompt):
    async with sem:
        for attempt in range(3):
            try:
                r = {"id": i, "response": await real_call(PROMPT, system_prompt), "error": None, "attempts": attempt+1, "model": MODEL, "prompt": PROMPT, "system_prompt": system_prompt }
                break
            except Exception as e:
                if attempt != 2:
                    await asyncio.sleep(2 ** attempt)
                r = {"id": i, "response": None, "error": f"error: {e}", "attempts": attempt+1, "model": MODEL, "prompt": PROMPT, "system_prompt": system_prompt}
        return r

start = time.time()
results = await asyncio.gather(
    *(limited_call(i, "You are a pirate.") for i in range(20)),
    *(limited_call(i, "You are a marine biologist.") for i in range(20, 40)),
)
print (results)
print(f"{time.time() - start:.1f}s")


[{'id': 0, 'response': 'Jellyfish', 'error': None, 'attempts': 1, 'model': 'qwen/qwen3.8-flash', 'prompt': 'Name one random animal. Reply with just the word.', 'system_prompt': 'You are a pirate.'}, {'id': 1, 'response': 'platypus', 'error': None, 'attempts': 1, 'model': 'qwen/qwen3.8-flash', 'prompt': 'Name one random animal. Reply with just the word.', 'system_prompt': 'You are a pirate.'}, {'id': 2, 'response': 'Octopus', 'error': None, 'attempts': 1, 'model': 'qwen/qwen3.8-flash', 'prompt': 'Name one random animal. Reply with just the word.', 'system_prompt': 'You are a pirate.'}, {'id': 3, 'response': 'Parrot', 'error': None, 'attempts': 1, 'model': 'qwen/qwen3.8-flash', 'prompt': 'Name one random animal. Reply with just the word.', 'system_prompt': 'You are a pirate.'}, {'id': 4, 'response': 'Octopus', 'error': None, 'attempts': 1, 'model': 'qwen/qwen3.8-flash', 'prompt': 'Name one random animal. Reply with just the word.', 'system_prompt': 'You are a pirate.'}, {'id': 5, 'respon

In [9]:
with open("chat_results.jsonl", "a") as f:
    for record in results:
        f.write(json.dumps(record) + ("\n"))

In [13]:
df = pd.read_json("chat_results.jsonl", lines = True)
df["clean"] = df["response"].str.lower().str.strip()
df.groupby("system_prompt")["clean"].value_counts()

system_prompt                clean    
You are a marine biologist.  octopus      12
                             platypus      3
                             jellyfish     2
                             otter         1
                             seahorse      1
                             cat           1
You are a pirate.            octopus       9
                             parrot        6
                             otter         2
                             jellyfish     1
                             platypus      1
                             shark         1
Name: count, dtype: int64